# 05 — EDA and histograms for run_02 (20k patients, all inpatient stays)

Input: `run_02/processed/Dataset_Tabular.csv` (notebook 2, cohort filter removed) + raw `encounters.csv` for admission reason.
Output: every figure and table saved to `run_02/eda/`. Nothing here trains a model.


In [ ]:
# Cell 1 — paths
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/genai_synthea/run_02"
DATA = f"{BASE}/processed/Dataset_Tabular.csv"
RAW  = f"{BASE}/raw/csv"
OUT  = f"{BASE}/eda"


In [ ]:
# Cell 2 — load, attach admission reason from raw encounters
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 70)

df = pd.read_csv(DATA, low_memory=False)
enc = pd.read_csv(f"{RAW}/encounters.csv", usecols=["Id", "ENCOUNTERCLASS", "DESCRIPTION", "REASONDESCRIPTION"])
enc = enc[enc["ENCOUNTERCLASS"] == "inpatient"].rename(columns={
    "Id": "hadm_id", "DESCRIPTION": "encounter_type", "REASONDESCRIPTION": "encounter_reason"})
df = df.merge(enc[["hadm_id", "encounter_type", "encounter_reason"]], on="hadm_id", how="left")
df["target"] = pd.to_numeric(df["target"], errors="coerce")
df = df.dropna(subset=["target"])

BIN  = [c for c in df.columns if c.startswith("snomed_")]
TEXT = [c for c in ["hpi", "social_history", "chief_complaint", "vitals", "labs", "conditions", "medications"] if c in df.columns]
print(f"Stays: {len(df):,} | patients: {df['subject_id'].nunique():,} | columns: {df.shape[1]} | condition flags: {len(BIN)}")
print(f"Encounter type matched for {df['encounter_type'].notna().mean():.1%} of stays")


In [ ]:
# Cell 3 — PLOS threshold candidates (the lab's rule is mean + 2*std)
mu, sd = df["target"].mean(), df["target"].std()
cands = {"mean+2std (lab)": mu + 2*sd, "p90": df["target"].quantile(.90), "p95": df["target"].quantile(.95)}
THR = cands["mean+2std (lab)"]
print(pd.DataFrame({"threshold_days": cands, "share_above": {k: (df["target"] > v).mean() for k, v in cands.items()}}).round(3))
df["is_plos"] = (df["target"] > THR).astype(int)
print(f"\nUsing {THR:.2f} days -> {df['is_plos'].sum():,} PLOS stays ({df['is_plos'].mean():.1%})")


In [ ]:
# Cell 4 — feature inventory (what is in the dataset)
def missing_rate(s):
    if s.dtype == object:
        return s.isna().mean() + s.astype(str).str.strip().str.lower().isin(["", "nan", "none", "[]"]).mean() * (1 - s.isna().mean())
    return s.isna().mean()
rows = []
for c in [c for c in df.columns if c not in BIN]:
    rows.append({"column": c, "dtype": str(df[c].dtype), "missing": missing_rate(df[c]), "n_unique": df[c].nunique()})
rows.append({"column": f"snomed_* ({len(BIN)} flags)", "dtype": "0/1", "missing": float(df[BIN].isna().mean().mean()), "n_unique": 2})
inv = pd.DataFrame(rows); inv.to_csv(f"{OUT}/feature_inventory.csv", index=False)
inv.round(3)


In [ ]:
# Cell 5 — LOS: linear, log, PLOS vs non-PLOS
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
ax[0].hist(df["target"], bins=100); ax[0].set_title("LOS (linear)")
ax[1].hist(df["target"], bins=100); ax[1].set_yscale("log"); ax[1].set_title("LOS (log count)")
for a in ax[:2]: a.axvline(THR, color="red", ls="--", label=f"threshold {THR:.1f} d"); a.set_xlabel("days"); a.legend()
b = np.linspace(0, df["target"].max(), 80)
ax[2].hist(df.loc[df.is_plos == 0, "target"], bins=b, alpha=.6, density=True, label="non-PLOS")
ax[2].hist(df.loc[df.is_plos == 1, "target"], bins=b, alpha=.6, density=True, label="PLOS")
ax[2].set_yscale("log"); ax[2].set_title("LOS by class (density)"); ax[2].legend()
fig.tight_layout(); fig.savefig(f"{OUT}/01_los.png", dpi=150); plt.show()
print(df["target"].describe(percentiles=[.5, .75, .9, .95, .99]).round(2).to_string())
top = df["target"].round(0).value_counts().head(5)
print("Most common LOS values (spikes = simulator artifacts):", top.to_dict())


In [ ]:
# Cell 6 — age, overall and by class
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(df["anchor_age"], bins=40); ax[0].set_title("Age at admission"); ax[0].set_xlabel("years")
for k, lab in [(0, "non-PLOS"), (1, "PLOS")]:
    ax[1].hist(df.loc[df.is_plos == k, "anchor_age"], bins=40, alpha=.6, density=True, label=lab)
ax[1].set_title("Age by class (density)"); ax[1].legend()
fig.tight_layout(); fig.savefig(f"{OUT}/02_age.png", dpi=150); plt.show()
print(df.groupby("is_plos")["anchor_age"].describe().round(1))


In [ ]:
# Cell 7 — demographics: counts and PLOS rate per category
cats = [c for c in ["gender", "race", "ethnicity", "marital_status"] if c in df.columns]
fig, ax = plt.subplots(1, len(cats), figsize=(5 * len(cats), 4))
for a, c in zip(np.atleast_1d(ax), cats):
    g = df.groupby(c)["is_plos"].agg(["size", "mean"]).sort_values("size", ascending=False)
    a.bar(g.index.astype(str), g["size"]); a.set_title(c); a.tick_params(axis="x", rotation=45)
    a2 = a.twinx(); a2.plot(g.index.astype(str), g["mean"], "ro-"); a2.set_ylabel("PLOS rate", color="r")
fig.tight_layout(); fig.savefig(f"{OUT}/03_demographics.png", dpi=150); plt.show()


In [ ]:
# Cell 8 — condition flags: sparsity and prevalence
df["n_conditions"] = df[BIN].sum(axis=1)
prev = df[BIN].mean().sort_values(ascending=False)
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].hist(df["n_conditions"], bins=range(0, int(df["n_conditions"].max()) + 2)); ax[0].set_title("Conditions per stay")
ax[1].hist(prev, bins=40); ax[1].set_yscale("log"); ax[1].set_title("Prevalence of each condition flag")
ax[1].set_xlabel("share of stays with the flag")
prev.head(30)[::-1].plot.barh(ax=ax[2]); ax[2].set_title("Top 30 condition flags")
fig.tight_layout(); fig.savefig(f"{OUT}/04_conditions.png", dpi=150); plt.show()
print(f"Flags present in <1% of stays: {(prev < .01).sum()} of {len(BIN)} | median conditions per stay: {df['n_conditions'].median():.0f}")


In [ ]:
# Cell 9 — which conditions are over-represented in long stays (starting point for modes)
lut = pd.read_csv(f"{RAW}/conditions.csv", usecols=["CODE", "DESCRIPTION"], dtype=str).drop_duplicates("CODE").set_index("CODE")["DESCRIPTION"]
cmp_ = pd.DataFrame({"plos_rate": df.loc[df.is_plos == 1, BIN].mean(), "nonplos_rate": df.loc[df.is_plos == 0, BIN].mean()})
cmp_ = cmp_[cmp_["plos_rate"] >= 0.02]
cmp_["ratio"] = cmp_["plos_rate"] / cmp_["nonplos_rate"].clip(lower=1e-3)
cmp_["diff"] = cmp_["plos_rate"] - cmp_["nonplos_rate"]
cmp_["desc"] = [lut.get(c.replace("snomed_", ""), "?") for c in cmp_.index]
cmp_ = cmp_.sort_values("diff", ascending=False); cmp_.to_csv(f"{OUT}/conditions_plos_vs_rest.csv")
fig, ax = plt.subplots(figsize=(10, 6))
t = cmp_.head(15)[::-1]
ax.barh(t["desc"].str[:45], t["nonplos_rate"], label="non-PLOS", alpha=.7)
ax.barh(t["desc"].str[:45], t["plos_rate"], label="PLOS", alpha=.7, height=.4)
ax.set_title("Conditions most over-represented in PLOS"); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/05_conditions_plos.png", dpi=150); plt.show()
cmp_.head(15).round(3)


In [ ]:
# Cell 10 — admission type / reason vs LOS (feature missing from Dataset_Tabular)
g = df.groupby("encounter_type")["target"].agg(n="size", median="median", p90=lambda x: x.quantile(.9))
g["plos_rate"] = df.groupby("encounter_type")["is_plos"].mean()
g = g[g["n"] >= 30].sort_values("plos_rate", ascending=False); g.to_csv(f"{OUT}/los_by_encounter_type.csv")
order = g.index.tolist()
fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot([df.loc[df.encounter_type == k, "target"] for k in order], vert=False, showfliers=False)
ax.set_yticklabels([o[:50] for o in order]); ax.axvline(THR, color="red", ls="--")
ax.set_xlabel("LOS (days)"); ax.set_title("LOS by encounter type (outliers hidden)")
fig.tight_layout(); fig.savefig(f"{OUT}/06_los_by_encounter_type.png", dpi=150); plt.show()
print(g.round(2).to_string())
print("\nTop encounter reasons among PLOS stays:")
print(df.loc[df.is_plos == 1, "encounter_reason"].value_counts().head(10).to_string())


In [ ]:
# Cell 11 — patients: repeat admissions
spp = df.groupby("subject_id").size()
pp = df.groupby("subject_id")["is_plos"].max()
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(spp.clip(upper=20), bins=range(1, 22)); ax.set_title("Inpatient stays per patient (capped at 20)")
fig.tight_layout(); fig.savefig(f"{OUT}/07_stays_per_patient.png", dpi=150); plt.show()
print(f"Patients: {len(spp):,} | median stays {spp.median():.0f} | max {spp.max()} | "
      f"patients with >=1 PLOS stay: {pp.mean():.1%}")


In [ ]:
# Cell 12 — separability check (did adding all patients + encounter type help?)
key_cols = [c for c in ["gender", "race", "ethnicity"] if c in df.columns] + BIN
for name, cols in [("conditions + demographics", key_cols), ("+ encounter type", key_cols + ["encounter_type"])]:
    key = df[cols].astype(str).agg("|".join, axis=1)
    neg = set(key[df.is_plos == 0])
    print(f"{name:28s}: PLOS stays with an identical non-PLOS profile = {key[df.is_plos == 1].isin(neg).mean():.1%}")


In [ ]:
# Cell 13 — text column coverage (Synthea vitals issue check)
print(pd.Series({c: 1 - missing_rate(df[c]) for c in TEXT}).round(3).rename("real coverage").to_string())
df.drop(columns=TEXT).to_parquet(f"{OUT}/stays_run02.parquet", index=False)
print("Saved figures and tables to", OUT)
